# Browse a database

In this tutorial, you connect a small database to Metanno and browse its tables from the JupyterLab sidebar,
without writing a query. The same steps work with your own database, whether it is a DuckDB file, PostgreSQL,
Spark or Trino.

!!! info "Before you start"

    Follow [Installation](../installation.md), including the `metanno-explorer` kernel. From now on, create the
    notebooks of these tutorials at the root of the JupyterLab file browser, where Metanno keeps its configuration
    and your views. This tutorial uses DuckDB, install its driver in the same environment:

    ```bash { data-md-color-scheme="slate" }
    pip install 'ibis-framework[duckdb]'
    ```

## Create a sample database

Run this cell in a notebook. It writes a few clinical notes to `clinic.duckdb`, next to the notebook, in French like the
notes [EDS-NLP](https://aphp.github.io/edsnlp/) pre-annotates in a later tutorial:

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd

rows = [
    (1, 10, "oncologie", "2026-01-05", "Cancer du sein, stade pT2N0M0."),
    (2, 10, "oncologie", "2026-02-10", "Pas de complication."),
    (3, 11, "cardiologie", "2026-01-20", "Fibrillation atriale."),
    (4, 12, "oncologie", "2026-03-02", "Tumeur cT3N1M0, pas de métastase."),
    (5, 12, "radiologie", "2026-03-15", "Mammographie sans lésion."),
    (6, 13, "cardiologie", "2026-04-01", "Insuffisance cardiaque."),
]
columns = ["note_id", "patient_id", "department", "note_date", "text"]
notes = pd.DataFrame(rows, columns=columns)
notes["note_date"] = pd.to_datetime(notes["note_date"])
path = Path("clinic.duckdb").resolve()
with duckdb.connect(str(path)) as connection:
    connection.register("new_notes", notes)
    connection.execute("CREATE OR REPLACE TABLE notes AS SELECT * FROM new_notes")
print(path)

Keep the printed path, you need it in the next step.

## Tell Metanno where it is

![The Metanno icon in the JupyterLab sidebar](../assets/images/tutorials/sidebar.png){ .float-right width="35" }

Click the Metanno icon in the left sidebar of JupyterLab, the two overlapping windows shown on the right. The
panel is empty for now: Metanno only lists the sources declared in its configuration file.

Click <span class="ui-button">Open configuration</span> at the top of the panel. It opens `metanno.yaml` in an
editor, created at the root of the file browser the first time, replace its content with the following, using
the path printed above:

```yaml
connections:
  clinic:
    url: duckdb:///home/you/clinic.duckdb
sources:
  clinic:
    connection: clinic
```

Save the file, then click the reload button <span class="ui-button">↻</span> next to
<span class="ui-button">Open configuration</span>.

`connections` says how to reach a database, and `sources` lists what the panel shows. Here a single source uses
the `clinic` connection, but a connection can serve several sources, for instance one per schema of the same
database.

## Explore a table

Under **Explore**, a `clinic` source now appears. Expand it and click `notes`: the table opens in a tab, with the
selected note on the right like in [Annotate a table](annotate-a-table.md).

This tab reads the database directly, without copying it. To keep things fast on large tables, it shows a preview
of at most 500 rows, in no particular order, and edits are disabled since they would change your database.

The boxes under the column headers filter the rows of this preview only. To search the whole table, type your
filters in these boxes, then click <span class="ui-button">Apply source filters</span> above the table: Metanno
sends them to the database and shows the matching rows. For instance, type `oncologie` under `department` and apply
it to the source.

??? tip "What else is in the tab toolbar?"

    | Button | What it does |
    | --- | --- |
    | View menu | Shows the same rows with another view, `table` or one of your own |
    | <span class="ui-button">Copy code</span> | Writes notebook code for the current query, the next tutorial uses it |
    | <span class="ui-button">Refresh</span> | Reloads the table, after the database changed or its connection failed |
    | Fullscreen | Fills the browser tab with the view, its URL reopens the same table |

    Right-click a table in the panel and choose **Open in new tab** to keep the current tab.

## What the panel lists

The panel has two parts:

- **Explore** lists your sources and their tables, and the saved views you will create in
  [Design your view](design-your-view.md)
- **Annotations** lists the files saved in the `views/stores` folder, next to `metanno.yaml`, such as the ones you
  will create in the next tutorial. Click one to reopen it, without opening its notebook

??? note "Other kinds of sources"

    Besides the tables of a database, a source can list the Parquet or Delta datasets of a folder, or the tables of
    one schema of a larger database. A connection can also be opened by your own Python function, for instance to
    read credentials:

    ```yaml
    connections:
      warehouse:
        factory: project_connections:connect
        kwargs:
          database: /data/warehouse.duckdb
    sources:
      oncology:
        connection: warehouse
        database: oncology
      exports:
        connection: warehouse
        kind: files
        format: parquet
        uri: /data/exports
    preview:
      rows: 1000
    ```

    - `catalog` and `database` select the catalogue and the database, or schema, holding the tables of a source
    - `kind: files` lists the datasets in the `uri` folder, local or on HDFS and S3 through a Spark connection
    - `factory` names a function, here `connect` in a `project_connections.py` file next to `metanno.yaml`, which
      returns an Ibis connection or a context manager giving one, and receives the `kwargs`
    - a factory connection whose `table()` lists the table files at each call, for instance the current files of a
      Delta table, can set `refresh_tables_on_view_open = True` on the connection it returns: Metanno then calls
      `table()` again each time a tab opens, so the tab sees the files written since the previous one
    - `preview.rows` changes the 500 rows shown by source tabs

    ```python { .no-exec }
    from contextlib import contextmanager

    import ibis


    @contextmanager
    def connect(database):
        backend = ibis.duckdb.connect(database)
        try:
            yield backend
        finally:
            backend.disconnect()
    ```

    Connections open the first time one of their tables is shown. Install the Ibis driver of each one, such as
    `ibis-framework[postgres]`, in the environment of the `metanno-explorer` kernel.

## Next

You found the notes you want to work on: [Pick documents](pick-documents.md) turns a query into an annotation
file.